In [1]:
%pip install tokenizers transformers torchmetrics


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import sklearn
import torch
import os
import pandas as pd
import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchmetrics

In [3]:
device = 'cpu'

In [4]:
if not os.path.exists('IMDB-Dataset.csv'):
  !wget -O IMDB-Dataset.csv -q "https://www.dropbox.com/scl/fi/0c7zc2adk1mgwgut5w80w/IMDB-Dataset.csv?rlkey=1drfg4zw36mhu32ndy2ihnygw&dl=1"

In [5]:
df = pd.read_csv('IMDB-Dataset.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [6]:
text = list(df['review'].str.replace('<br />',''))
labels = np.array(df['sentiment'].map({'negative':0,'positive':1}))

In [7]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Example of how to tokenize text:


In [8]:
seq = text[0][:10]
seq

'One of the'

In [9]:
token_ids = tokenizer(seq)['input_ids']
token_ids

[101, 1448, 1104, 1103, 102]

In [10]:
tokenizer.decode(token_ids+[0,0,0])

'[CLS] One of the [SEP] [PAD] [PAD] [PAD]'

# Bag of Words Model

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(text, labels, test_size=0.1, stratify=labels, random_state=42)

vectorizer = TfidfVectorizer(max_features=1000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

mlp = MLPClassifier(hidden_layer_sizes=(100,), random_state=42)
mlp.fit(X_train_tfidf, y_train)

train_acc = mlp.score(X_train_tfidf, y_train)
test_acc = mlp.score(X_test_tfidf, y_test)
print(f"BoW Model - Train Accuracy: {train_acc:.4f}, Test Accuracy: {test_acc:.4f}")

BoW Model - Train Accuracy: 1.0000, Test Accuracy: 0.8654


# RNN Model (GRU)

In [ ]:
max_length = 100

def process_texts(texts, tokenizer, max_length):
    processed = []
    for text in texts:
        token_ids = tokenizer.encode(text, add_special_tokens=False) 
        if len(token_ids) > max_length:
            start_idx = np.random.randint(0, len(token_ids) - max_length)
            token_ids = token_ids[start_idx:start_idx + max_length]
        else:
            token_ids += [0] * (max_length - len(token_ids))
        processed.append(token_ids)
    return processed


In [16]:

processed_X_train = process_texts(X_train, tokenizer, max_length)
processed_X_test = process_texts(X_test, tokenizer, max_length)

train_sequences = torch.tensor(processed_X_train, dtype=torch.long)
test_sequences = torch.tensor(processed_X_test, dtype=torch.long)
train_labels = torch.tensor(y_train, dtype=torch.float)
test_labels = torch.tensor(y_test, dtype=torch.float)

batch_size = 32
train_dataset = torch.utils.data.TensorDataset(train_sequences, train_labels)
test_dataset = torch.utils.data.TensorDataset(test_sequences, test_labels)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)


Token indices sequence length is longer than the specified maximum sequence length for this model (765 > 512). Running this sequence through the model will result in indexing errors


In [18]:
class GRUModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.gru = nn.GRU(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        
    def forward(self, x):
        embedded = self.embedding(x)
        output, _ = self.gru(embedded)
        last_output = output[:, -1, :]  
        return torch.sigmoid(self.fc(last_output).squeeze())

model = GRUModel(
    vocab_size=tokenizer.vocab_size,
    embedding_dim=100,
    hidden_dim=100,
    num_layers=3
)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
criterion = nn.BCELoss()


In [20]:

device = 'mps' if torch.cuda.is_available() else 'cpu'
model.to(device)
for epoch in range(10):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * inputs.size(0)
    avg_loss = total_loss / len(train_loader.dataset)
    
    # Evaluate
    model.eval()
    for name, loader in [("Train", train_loader), ("Test", test_loader)]:
        preds, truths = [], []
        with torch.no_grad():
            for inputs, labels in loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                preds.extend((outputs >= 0.5).float().cpu().numpy())
                truths.extend(labels.cpu().numpy())
        acc = sklearn.metrics.accuracy_score(truths, preds)
        print(f"Epoch {epoch+1} {name} Acc: {acc:.4f}", end=' | ')
    print()

Epoch 1 Train Acc: 0.8543 | Epoch 1 Test Acc: 0.7948 | 
Epoch 2 Train Acc: 0.9043 | Epoch 2 Test Acc: 0.8156 | 
Epoch 3 Train Acc: 0.9175 | Epoch 3 Test Acc: 0.8128 | 
Epoch 4 Train Acc: 0.9455 | Epoch 4 Test Acc: 0.8178 | 
Epoch 5 Train Acc: 0.9583 | Epoch 5 Test Acc: 0.8150 | 
Epoch 6 Train Acc: 0.9636 | Epoch 6 Test Acc: 0.8130 | 
Epoch 7 Train Acc: 0.9827 | Epoch 7 Test Acc: 0.8084 | 
Epoch 8 Train Acc: 0.9781 | Epoch 8 Test Acc: 0.8018 | 
Epoch 9 Train Acc: 0.9867 | Epoch 9 Test Acc: 0.7990 | 
Epoch 10 Train Acc: 0.9830 | Epoch 10 Test Acc: 0.8078 | 
